# Project 2

# Correlation Between League of Legends Solo Queue vs. Pick Rate in League of Legend's Professional Scene?

For this project I wanted to see if there was a relationship between solo queue and professional play. Playing League of Legends at a high level, and playing with pros on the solo queue ladder, I realized that their performances on stage is very different than when they play in solo queue. I wanted to explore if there was any relationship between the two at all. To do this, I analyzed two different Kaggle datasets:

**Information on League of Legend's World's championship 2022:** https://www.kaggle.com/datasets/ilyadziamidovich/league-of-legends-world-championship-2022

**Information on League of Legend's champions for patch 12.18 (The patch the world championship was played on):** https://www.kaggle.com/datasets/vivovinco/league-of-legends-champion-stats

Similar to my experience with pro players, my hypothesis for the project was:

Hypothesis: Champion winrate in solo queue does not affect professional's picking that champion in a professional setting. 

Let's get to exploring :DD

In [1]:
# Some Library Imports
import pandas as pd
import plotly.express as px
from IPython.display import HTML

In [2]:
# Creating data frame for League of Legends Solo Queue and Worlds Stats
df_solo = pd.read_csv("solo_queue.csv", sep = ";") 
df_worlds = pd.read_csv("worlds.csv")

# Cleaning up Dataset One: Solo Queue!

The first step in the project was cleaning up the data set. I decided to go with solo queue cause thats what I imported first :^).

The process involved picking the columns I wanted, and making sure that the numbers I had were usuable for when I wanted to convert it into visualization. A big hiccup during this process was that the dataset had seperated champion's by positions as well. This meant that hypothetically, if a champion were played in two roles like top and mid, that champ would have a seperate row for those two positions. To solve this, I combined the names, and took the averages of the winrates to make sure no champions were duplicated.

In [3]:
#Take the information we want for solo queue
df_solo_name = df_solo["Name"]
df_win_rate = df_solo["Win %"]
df_solo_new = pd.concat([df_solo_name, df_win_rate], axis = 1)

def clean_percent(x):    #Format to turn values into floats
    if "%" in x:
        x = x.replace("%","")
    return round(float(x),2)
df_solo_new["Win %"] = df_solo_new["Win %"].apply(clean_percent)

df_solo_new = df_solo_new.groupby(by = ["Name"]).mean().reset_index() #Combines same names and averages out the win percentages
df_solo_new = df_solo_new.round(2) #Rounds for better visualization

#Looking at highest and Lowest Winrate to set my percentages for table
df_solo_new.loc[df_solo_new["Win %"].idxmax(), ["Name", "Win %"]], df_solo_new.loc[df_solo_new["Win %"].idxmin(), ["Name", "Win %"]]


(Name     Singed
 Win %     53.22
 Name: 113, dtype: object,
 Name      Zeri
 Win %    44.19
 Name: 156, dtype: object)

### Data Visualization for Solo Queue

Below, I wanted to see a see a sample of 10 champs, and their win rates. Very cool :D

In [9]:
#Bar Chart Visualization for Champions WR in solo Queue
solo_bar = px.bar(df_solo_new.sample(10), x = "Name", y = "Win %", title = "Champion Winrate in Solo Queue", height = 300)
solo_bar.update_yaxes(range =[40,60],dtick = 5)
#solo_bar.show()
HTML(solo_bar.to_html(include_plotlyjs="cdn", full_html=False))

# Cleaning up Dataset Two: Pro Play (Worlds)!

Same idea as before, cleaning up the dataset, and picking out which columns would be useful to me. One interesting step was that the data did not have a total count of the games, so I had to create a variable for that. From there, I created a column that got the percent a champion was played.

In [11]:
# Worlds Data Cleaning
df_worlds_champion = df_worlds["champion"]
total_games = df_worlds["sum_total"].sum() #total to divide and get pick rate % later
df_worlds_total = df_worlds["sum_total"]
df_worlds_new = pd.concat([df_worlds_champion, df_worlds_total],axis = 1)

def percent_divide(x):
    pick_percent = (float(x) / total_games) * 100
    return pick_percent.round(2)

df_worlds_new["pick %"] = df_worlds_new["sum_total"].apply(percent_divide)

#Looking at Max in Min to figure out range visualization for later
df_worlds_new.loc[df_worlds_new["pick %"].idxmax(), ["champion", "pick %"]], df_worlds_new.loc[df_worlds_new["pick %"].idxmin(), ["champion", "pick %"]]


(champion    Sylas
 pick %       3.93
 Name: 0, dtype: object,
 champion    Karthus
 pick %         0.08
 Name: 88, dtype: object)

### Data Visualization for Pro Play (Worlds)

Same idea as before. Very neat :O

In [15]:
#Worlds Pick Rate Visualization
worlds_bar = px.bar(df_worlds_new.sample(10), x = "champion", y = "pick %", title = "Champion Pick Rate at Worlds", height = 500)
worlds_bar.update_yaxes(range = [0,5], dtick = 0.5)
#worlds_bar.show()
HTML(worlds_bar.to_html(include_plotlyjs="cdn", full_html=False))

### Working With the Two Dataframes Together

Finally, we get to working with the two dataframes together. I first began with renaming the columns in both dataframes to make sure they would match. By doing so, it made the merge a lot easier. One thing to note is that the there were 161 champions in 2022 (and reflected in the solo queue dataset), but in the combined data, there are only 108. There are some champions missing! I will address that later :^) 

In [16]:
#Combining Both DataFrames and Creating Visualization

df_worlds_new = df_worlds_new.rename(columns = {"champion":"Champion","pick %":"Pick %"}) #Rename both for better merging :D
df_solo_new = df_solo_new.rename(columns = {"Name": "Champion"})

df_combined = df_solo_new.merge(df_worlds_new, how = "right")
df_combined = df_combined.drop("sum_total", axis = 1) #Dropping sum_total because it is not needed for visualization
df_combined

,Champion,Win %,Pick %
0,Sylas,50.06,3.93
1,Sejuani,50.65,3.86
2,Azir,44.56,3.78
3,Aatrox,49.95,3.62
4,Aphelios,46.51,3.38
...,...,...,...
103,Singed,53.22,0.08
104,Sion,50.66,0.08
105,Teemo,50.02,0.08
106,Tryndamere,50.38,0.08


### Combined Data Visualization

In [17]:
#Combined Data Visualization
combined_scatter = px.scatter(df_combined, x = "Pick %", y = "Win %", title = "Relationship between Champion Solo Queue Winrate and Champion Pick Rate at Worlds", hover_name = "Champion")
#combined_scatter.show()
HTML(combined_scatter.to_html(include_plotlyjs="cdn", full_html=False))

### NO CORRELATION :(

![My Photo](images/myphoto.jpg)



Points are all over the place, and there in no trend this scatterplot.

### Conclusion

The scatterplot results support my hypothesis. League of Legends is a team game, and in solo queue where the only form of communication is through typing and pings, many champions that require teamwork tend to be worse. For example, Azir has an atrocious winrate of 44.56% win rate in solo queue, but a rather high pick rate of 3.78% at worlds. Azir is a champion that does exceptionally well when players are able to communicate through microphone, a accessory that is used in professional play. 

On top of this, 53 champions of the total 161 were not even played at worlds.

While they are the same game, world's meta and solo queue meta are completely different, and the stats should not be intermingled among each other.